# Defringe (green → purple cast) — Colab GPU runner

Runs the ONNX defringe model over a video on a Colab GPU and writes the result back to Google Drive.
The model is the **green → purple cast** pipeline exported from the tuner (`cast_defringe.onnx`):
uint8 RGB frames in → uint8 RGB out, dynamic N/H/W, all colour-space math baked into the graph.
Uses **batched inference + threaded decode/encode** so the GPU stays fed (a serial batch-1 loop
leaves the GPU ~15% utilised).

**Colour handling:** the re-encode now controls the YUV↔RGB matrix and **tags the output** so it
doesn't shift colour. The sample source is HD but tagged BT.601 (`bt470bg`); the default
`COLOR_MODE="bt709"` converts + tags the output as standard HD BT.709 so every player reads it
correctly (see the Config cell).

**Setup (do this first):**
1. `Runtime → Change runtime type → GPU` (T4 works; A100 is faster).
2. Upload `cast_defringe.onnx` and your video to a folder on your Google Drive.
3. Edit the **Config** cell, then run all cells top-to-bottom.

### 1. Install dependencies
`ffmpeg` is preinstalled on Colab; we just need the GPU build of ONNX Runtime.

In [ ]:
!pip -q install onnxruntime-gpu
import onnxruntime as ort
print("onnxruntime", ort.__version__, "| providers:", ort.get_available_providers())
!ffmpeg -version | head -n1

### 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3. Config — edit these

In [ ]:
# Drive lives at /content/drive/MyDrive/...
MODEL_GDRIVE  = "/content/drive/MyDrive/defringe/cast_defringe.onnx"
VIDEO_GDRIVE  = "/content/drive/MyDrive/defringe/sanguo-ep01-10min.mp4"
OUTPUT_GDRIVE = "/content/drive/MyDrive/defringe/sanguo-defringed.mp4"

DURATION = None      # seconds from the start; None = the WHOLE video
BATCH    = 16        # frames per GPU call. Bump to 16/32 if VRAM allows (each 1080p frame ~ small)
KEEP_AUDIO = True

# Output quality / codec:
#   "crf"         -> H.264 4:2:0 at CRF below (lossy but small; default). NOTE 4:2:0 halves
#                    chroma — the very channel defringe edits — so subtle fringe fixes soften a bit.
#   "lossless444" -> H.264 4:4:4 -qp 0: mathematically lossless, ALL chroma kept (~10x bigger;
#                    needs a 4:4:4-capable player — VLC/ffmpeg fine, QuickTime/Safari may not).
#   "losslessrgb" -> libx264rgb -qp 0: bit-exact RGB, no YUV conversion at all (~20x bigger,
#                    truest archival, least universally playable).
# Rough sizes for 10 min @1080p: crf ~1 GB, lossless444 ~12 GB, losslessrgb ~22 GB.
QUALITY = "crf"
CRF     = 16         # only used when QUALITY == "crf" (lower = better/larger)

# Colour handling for the re-encode. A file carries a YUV<->RGB *matrix* (BT.601 vs
# BT.709), primaries, transfer and range; if the output is untagged or uses a
# different matrix than players assume, colours shift. This sample source is HD but
# tagged BT.601 (bt470bg), and most players assume HD == BT.709 -> shift. Modes:
#   "bt709"    -> convert + tag the output as BT.709 limited (RECOMMENDED; makes a
#                 standard HD-709 file every player reads correctly). Decoding the
#                 601 source to RGB then encoding as 709 == ffmpeg's colormatrix=bt601:bt709.
#   "preserve" -> keep the source's exact colour space + tags (probe-driven). Use when
#                 the source is already correctly tagged (e.g. a real BT.709 master).
#   "none"     -> legacy: no matrix control, untagged output (this is the glitch).
COLOR_MODE = "bt709"

### 4. Stage files on local disk
Drive's FUSE mount is slow for ffmpeg's reads, so copy locally first.

In [ ]:
import os, shutil, time
os.makedirs('/content/work', exist_ok=True)
MODEL        = '/content/work/model.onnx'
VIDEO        = '/content/work/input' + os.path.splitext(VIDEO_GDRIVE)[1]
OUTPUT_LOCAL = '/content/work/output.mp4'
print('copying from Drive to local disk...')
shutil.copy(MODEL_GDRIVE, MODEL); shutil.copy(VIDEO_GDRIVE, VIDEO)
print('model:', os.path.getsize(MODEL)//1024, 'KB |', 'video:', os.path.getsize(VIDEO)//(1024*1024), 'MB')

### 5. Create the session & confirm the GPU is active

In [ ]:
import onnxruntime as ort
PROVIDERS = ['CUDAExecutionProvider', 'CPUExecutionProvider']
_sess = ort.InferenceSession(MODEL, providers=PROVIDERS)
print('using:', _sess.get_providers())
if 'CUDAExecutionProvider' not in _sess.get_providers():
    print('\u26a0\ufe0f  CUDA EP not active -> running on CPU (much slower).')
    print('   Fix: Runtime -> Change runtime type -> GPU, then re-run the install cell.')

### 6. Run the defringe pipeline (batched + threaded)
Decoder thread \u2192 batched ONNX inference \u2192 encoder thread, with a live progress bar.

In [ ]:
import subprocess, json, threading, queue, time
import numpy as np
from tqdm.auto import tqdm

def probe(path):
    out = subprocess.check_output(['ffprobe','-v','error','-select_streams','v:0',
        '-show_entries','stream=width,height,r_frame_rate,nb_frames,'
        'color_space,color_primaries,color_transfer,color_range','-of','json', path])
    s = json.loads(out)['streams'][0]
    num, den = s['r_frame_rate'].split('/')
    s['_fps'] = float(num)/float(den)
    return s

# ffprobe color_space tag -> swscale matrix name (for the scale filter's out_color_matrix)
_MTX = {'bt709':'bt709','bt470bg':'bt601','smpte170m':'bt601','bt601':'bt601',
        'fcc':'fcc','smpte240m':'smpte240m','bt2020nc':'bt2020','bt2020_ncl':'bt2020'}
_UNK = ('unknown','unspecified','reserved',None,'')

def encode_color(s, mode):
    """-> (colorspace, primaries, transfer, swscale_matrix, range). Decode stays plain
    (reads the source's own tags -> faithful RGB); only the *encode* is controlled + tagged."""
    H = int(s['height'])
    cs = s.get('color_space');     cr = s.get('color_range')
    cp = s.get('color_primaries'); ct = s.get('color_transfer')
    if cs in _UNK: cs = 'bt709' if H >= 720 else 'smpte170m'     # ffmpeg's own HD/SD guess
    if cp in _UNK: cp = cs
    if ct in _UNK: ct = cs
    rng = 'pc' if cr in ('pc','full','jpeg') else 'tv'
    if mode == 'bt709':
        return 'bt709','bt709','bt709','bt709','tv'              # convert+tag as standard HD-709
    return cs, cp, ct, _MTX.get(cs,'bt709'), rng                 # preserve the source exactly

def out_video_args(s, quality, crf, color_mode):
    """Encoder video-filter + codec args for the chosen quality & colour.
       quality: 'crf' (lossy 4:2:0) | 'lossless444' (lossless, all chroma) | 'losslessrgb' (bit-exact RGB)."""
    tag = None if color_mode == 'none' else encode_color(s, color_mode)
    if quality == 'losslessrgb':                                 # RGB: no YUV matrix at all
        codec = ['-c:v','libx264rgb','-qp','0','-pix_fmt','rgb24']
        vf = [] if tag is None else ['-vf', f'setparams=color_primaries={tag[1]}:color_trc={tag[2]}']
        qd = 'lossless RGB (bit-exact)'
    else:
        if quality == 'lossless444':
            codec = ['-c:v','libx264','-qp','0','-pix_fmt','yuv444p']; pix='yuv444p'; qd='lossless 4:4:4'
        else:                                                    # 'crf'
            codec = ['-c:v','libx264','-crf',str(crf),'-pix_fmt','yuv420p']; pix='yuv420p'; qd=f'crf {crf} (4:2:0, lossy)'
        if tag is None:
            vf = []
        else:
            cs,cp,ct,mtx,rng = tag
            vf = ['-vf', f'scale=in_range=pc:out_range={rng}:out_color_matrix={mtx},'
                         f'format={pix},setparams=range={rng}:colorspace={cs}:'
                         f'color_primaries={cp}:color_trc={ct}']
    return vf, codec + ['-movflags','+faststart'], qd

def defringe_video(inp, out, model, duration=None, batch=8, crf=16, audio=True,
                   color_mode='bt709', quality='crf'):
    s = probe(inp); W, H, rate, fps = int(s['width']), int(s['height']), s['r_frame_rate'], s['_fps']
    nb = int(s.get('nb_frames') or 0)
    total = int(round(duration*fps)) if duration else (nb or None)
    fsz = W*H*3

    enc_vf, vcodec, qdesc = out_video_args(s, quality, crf, color_mode)
    cdesc = 'untagged' if color_mode=='none' else \
            f"{s.get('color_space') or '?'}/{s.get('color_range') or '?'} -> {color_mode}"
    print(f'{W}x{H} @ {fps:.3f} fps |', f'first {duration}s' if duration else 'full',
          f'| batch {batch} | {qdesc} | colour: {cdesc}')

    # decode stays plain: reads the source's own colour tags -> faithful RGB (same as the tuner)
    dec = ['ffmpeg','-v','error'] + (['-t',str(duration)] if duration else []) \
        + ['-i', inp, '-f','rawvideo','-pix_fmt','rgb24','-']
    enc = ['ffmpeg','-v','error','-y','-f','rawvideo','-pix_fmt','rgb24','-s',f'{W}x{H}','-r',rate,
           '-color_range','pc','-i','-']                          # tell ffmpeg the raw RGB is full-range
    if audio:
        enc += (['-t',str(duration)] if duration else []) + ['-i', inp, '-map','0:v:0','-map','1:a:0?','-c:a','aac','-shortest']
    enc += enc_vf + vcodec + [out]

    dp = subprocess.Popen(dec, stdout=subprocess.PIPE)
    ep = subprocess.Popen(enc, stdin=subprocess.PIPE)
    sess = ort.InferenceSession(model, providers=PROVIDERS)
    iname = sess.get_inputs()[0].name
    q_in, q_out = queue.Queue(maxsize=4), queue.Queue(maxsize=4)

    def reader():
        buf_b = []
        while True:
            buf = dp.stdout.read(fsz)
            if len(buf) < fsz: break
            buf_b.append(np.frombuffer(buf, np.uint8).reshape(H, W, 3))
            if len(buf_b) == batch:
                q_in.put(np.stack(buf_b, 0)); buf_b = []
        if buf_b: q_in.put(np.stack(buf_b, 0))
        q_in.put(None)

    def writer():
        while True:
            it = q_out.get()
            if it is None: break
            ep.stdin.write(it)

    rt = threading.Thread(target=reader, daemon=True); rt.start()
    wt = threading.Thread(target=writer, daemon=True); wt.start()

    bar = tqdm(total=total, unit='frame'); n = 0; t0 = time.perf_counter()
    while True:
        b = q_in.get()
        if b is None: break
        y = sess.run(None, {iname: b})[0]   # uint8 NHWC in -> uint8 NHWC out (norm + per-frame lighting-tone on-device)
        q_out.put(y.tobytes()); n += len(b); bar.update(len(b))
        bar.set_postfix_str(f'{(time.perf_counter()-t0)/n*1000:.0f} ms/frame')
    q_out.put(None); bar.close()
    wt.join()                                  # drain ALL frames before closing the encoder's stdin
    ep.stdin.close(); dp.stdout.close(); ep.wait(); dp.wait()
    print(f'done: {n} frames in {time.perf_counter()-t0:.0f}s')
    return out

defringe_video(VIDEO, OUTPUT_LOCAL, MODEL, duration=DURATION, batch=BATCH, crf=CRF,
               audio=KEEP_AUDIO, color_mode=COLOR_MODE, quality=QUALITY)

### 7. Save the result back to Google Drive

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_GDRIVE), exist_ok=True)
shutil.copy(OUTPUT_LOCAL, OUTPUT_GDRIVE)
print('saved ->', OUTPUT_GDRIVE, '(', os.path.getsize(OUTPUT_GDRIVE)//(1024*1024), 'MB )')